# Model Results

Fits and evalutes the Bayesian model.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, RANDOM_SEED, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model.model import BayesianPoissonModel, BayesianPoissonModelConfig, SVIConfig

/Users/astonchan/Desktop/Academics Folder/UC Irvine/Senior Year/Spring Quarter/COMPSCI 179 – Algorithms for Probabilistic and Deterministic Graphical Models/Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation

### Load Training and Testing Data

In [4]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 3040 matches
Test: 760 matches


### Bayesian Model

In [5]:
bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = False,
        ablate_defense = False,
        ablate_home_team_advantage = False,
        num_posterior_samples_per_match = 100,
    )
).fit(train_df)
bayesian_model_preds = bayesian_model.predict(test_df)

Step 100: Loss = 9148.932
Step 200: Loss = 9078.869
Step 300: Loss = 9072.959
Step 400: Loss = 9068.296
Step 500: Loss = 9064.379
Step 600: Loss = 9065.652
Step 700: Loss = 9064.019
Step 800: Loss = 9066.084
Step 900: Loss = 9066.602
Step 1000: Loss = 9067.518
Step 1100: Loss = 9067.047
Step 1200: Loss = 9067.893
Step 1300: Loss = 9063.794
Step 1400: Loss = 9065.432
Step 1500: Loss = 9066.597
Step 1600: Loss = 9071.470
Step 1700: Loss = 9070.947
Step 1800: Loss = 9063.042
Step 1900: Loss = 9068.942
Step 2000: Loss = 9065.840
Step 2100: Loss = 9063.655
Step 2200: Loss = 9065.275
Step 2300: Loss = 9074.979
Step 2400: Loss = 9073.610
Step 2500: Loss = 9067.943
Step 2600: Loss = 9074.875
Step 2700: Loss = 9066.876
Step 2800: Loss = 9065.068
Step 2900: Loss = 9070.057
Step 3000: Loss = 9069.724
Step 3100: Loss = 9067.835
Step 3200: Loss = 9067.330
Step 3300: Loss = 9066.399
Step 3400: Loss = 9063.573
Step 3500: Loss = 9068.618
Step 3600: Loss = 9072.090
Step 3700: Loss = 9066.233
Step 3800:

In [6]:
bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.637752,0.207890,0.154358
1,West Ham,Aston Villa,A,0.385377,0.247519,0.367104
2,Nott'm Forest,Bournemouth,D,0.445604,0.239155,0.315242
3,Newcastle,Southampton,H,0.529534,0.245736,0.224730
4,Everton,Brighton,A,0.426446,0.275180,0.298374


In [7]:
pred_labels = bayesian_model_preds[["ProbHomeWin",
                                    "ProbDraw",
                                    "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                          "ProbDraw": "D",
                                                                          "ProbAwayWin": "A"})
accuracy = (pred_labels == bayesian_model_preds["Result"]).mean()
print(f"Bayesian Model Accuracy: {accuracy:.3f}")

Bayesian Model Accuracy: 0.461


### Save Evaluation Results

In [8]:
model_table_output_directory = (TABLES_DIR / "model")
model_table_output_directory.mkdir(parents = True, exist_ok = True)

bayesian_model_preds.to_csv(model_table_output_directory / "bayesian_model_predictions.csv",
                            index = False)